# Taller LangChain + LangSmith: puntos 6 al 10 con Gemma 3 local

Adaptación del notebook original. **Ejecutar en VS Code** con el kernel de `.venv` y Ollama activo.

Usamos el modelo **`gemma3:1b`** ya instalado. Para evitar problemas de *native tool calling* con esa versión, las herramientas se orquestan mediante grafos de LangGraph; Gemma escribe las respuestas. La distinción frente a `create_agent` autónomo se explica en `README.md`.

## Preparación (una vez por entorno)

Primero abre la carpeta completa en VS Code, crea `.venv` e instala `requirements.txt` como se describe en `README.md`. Edita `.env` con `OLLAMA_MODEL=gemma3:1b` y, si deseas registrar trazas, con la clave y el agente de LangSmith. Reinicia el kernel si modificas `.env`.

La siguiente celda sirve para verificar la conexión directa a Ollama.

In [ ]:
from configuracion import crear_modelo, MODELO_NOMBRE

modelo = crear_modelo()
print("Modelo local:", MODELO_NOMBRE)
print(modelo.invoke("Di hola en una frase en español.").content)

## 6. Agente de clima y operaciones aritméticas

La herramienta `calcular` ejecuta aritmética segura mediante el árbol sintáctico de Python (`ast`): suma, resta, multiplicación, división, potencias y paréntesis. No utiliza `eval`. `clima` sigue disponible (simulado con `CLIMA_REAL=false`). Gemma redacta la respuesta y LangGraph orquesta el flujo.


In [ ]:
from herramientas import clima, calcular
from agentes import consultar_herramientas

for expresion in ["8+4", "10-3", "20/4", "2**5", "(3+4)*2"]:
    print(expresion, "=", calcular.invoke({"expresion": expresion}))

resultado_6 = consultar_herramientas("¿Qué clima hace en Cali? ¿Cuánto es 23.5 por 4 y 2^3?")
print("\nHerramientas usadas:")
for dato in resultado_6["resultados_herramientas"]:
    print("-", dato)
print("\nRespuesta final:", resultado_6["respuesta"])


**LangSmith:** revisa la traza `Punto_6_agente_clima_calculadora` y sus nodos `usar_herramientas` y `responder_con_gemma`. Las trazas requieren `LANGSMITH_TRACING=true` y una API key válida.


## 7. RAG de preguntas frecuentes de Mercado Libre Colombia con BM25

Se utiliza una copia **local y resumida** de 12 preguntas frecuentes tomadas de páginas públicas oficiales de Mercado Libre Colombia, con URLs de origen en `datos/faq_mercadolibre_colombia.json`. La base no se actualiza automáticamente: consulta el sitio oficial para confirmar cambios. BM25 busca fragmentos; Gemma responde usando únicamente el contexto.


In [ ]:
from agentes import consultar_rag

resultado_7 = consultar_rag("¿Puedo pagar contra entrega?")
print("CONTEXTO RECUPERADO:\n", resultado_7["contexto"])
print("\nRESPUESTA:\n", resultado_7["respuesta"])

In [ ]:
# Pregunta deliberadamente fuera de las FAQ: no debe inventar información privada.
resultado_sin_evidencia = consultar_rag("¿Cuánto gana el gerente de Mercado Libre?")
print(resultado_sin_evidencia["respuesta"])


## 8. Traductor + `@traceable`

El notebook original usa `@traceable` en el punto 8; el **traductor es un ejercicio adicional solicitado** y usa esa misma funcionalidad. La función `traducir` está decorada para trazar entrada, salida y pasos del pipeline.

In [ ]:
from agentes import traducir

print(traducir(
    texto="El presupuesto del proyecto está listo para revisión.",
    origen="español",
    destino="inglés",
))
print(traducir(
    texto="The construction schedule needs an update.",
    origen="inglés",
    destino="español",
))

## 9. Evaluación con LangSmith

El programa `evaluar.py` crea un dataset independiente para las FAQ de Mercado Libre y utiliza una comprobación básica de coincidencia textual. No evalúa fidelidad ni exhaustividad. Necesita API key y conexión a Internet.


In [ ]:
import os
from configuracion import RAIZ

if os.getenv("LANGSMITH_API_KEY"):
    from evaluar import main as evaluar_en_langsmith
    evaluar_en_langsmith()
else:
    print("Configura LANGSMITH_API_KEY en .env y reinicia el kernel para evaluar.")

## 10. Memoria conversacional con `InMemorySaver`

Con el mismo `thread_id`, el historial permanece entre dos invocaciones dentro de una sesión de Python. No es memoria persistente en disco.

In [ ]:
from memoria import demostracion_memoria

print(demostracion_memoria())

### Alternativa opcional: `create_agent` nativo

Si el profesor exige exactamente la llamada `create_agent` del punto 6, revisa `agente_nativo_opcional.py` y el apartado 8 de `README.md`. **Necesita descargar** `lukaspetrik/gemma3-tools:1b`; no es el mismo archivo de modelo `gemma3:1b` ya instalado. No se ha validado su comportamiento en tu máquina.